<a href="https://colab.research.google.com/github/rahulsapkota01/Rahul_INFO4670_Fall2026/blob/main/INFO4670_Assignment2_Northgate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# INFO 4670 / 4760 — Assignment 2 (Framework)
### Cleaning & Integrating the Northgate Data

Fill in each **# TODO** cell with your code, then run the **✅ Check** cell under it to see if it passes. Work top to bottom. When you're done, run the whole notebook once (Runtime → Run all), make sure it runs cleanly, and submit your **GitHub link**.

- Do every fix on a **copy** — never overwrite the raw files.
- The Week 5 Guided notebook shows every technique you need.
- You're graded on correct operations **and** justified decisions (see the rubric).

## Setup — load the three files (given)

In [ ]:
import pandas as pd, numpy as np, os
try:
    students = pd.read_csv("student_records.csv")
except FileNotFoundError:
    from google.colab import files
    print("Upload student_records.csv, course_enrollments.csv, weekly_activity.csv")
    files.upload()
    students = pd.read_csv("student_records.csv")
enroll   = pd.read_csv("course_enrollments.csv")
activity = pd.read_csv("weekly_activity.csv")
# Golden rule: work on copies, never overwrite the raw files.
print("students", students.shape, "| enroll", enroll.shape, "| activity", activity.shape)

students (2027, 12) | enroll (8088, 4) | activity (32000, 4)


## Part A · Clean student_records

### A1 · Missing values
Find how many values are missing in `study_hours_reported` and store the count as **`n_missing_study`**. Then, in the markdown cell after your code, say in 1–2 sentences which of Han's methods you would use to handle it and why.
*Hint:* `.isna().sum()`

In [ ]:
# TODO: set n_missing_study to the number of blank study_hours_reported values
n_missing_study = students["study_hours_reported"].isna().sum()

**Your Justification (102 sentences):**I would keep the missing study hours and report the available sample size. Filling them with the mean could make the data misleading because the missing values may not be random.



In [ ]:
# ✅ Check
try:
    assert n_missing_study == 255
    print("✅ A1 correct — 255 missing (n = 1772 present)")
except Exception:
    print("❌ A1 not yet — set n_missing_study to the count of blank study_hours_reported")

✅ A1 correct — 255 missing (n = 1772 present)


### A2 · Inconsistent categories
Standardize the `housing` column into its three real groups and store the result as a new column **`students["housing_clean"]`**.
*Hint:* `.str.strip().str.lower().map({...})`

In [ ]:
# TODO: create students["housing_clean"] with exactly 3 standardized groups
students["housing_clean"] = (
    students["housing"]
    .str.strip()
    .str.lower()
    .map({
        "on-campus": "on-campus",
        "on campus": "on-campus",
        "oncampus": "on-campus",
        "off-campus": "off-campus",
        "off campus": "off-campus",
        "offcampus": "off-campus",
        "with family": "with family",
        "with_family": "with family"
    })
)

In [ ]:
# ✅ Check
try:
    assert students["housing_clean"].nunique() == 3
    print("✅ A2 correct — 3 groups:", {k:int(v) for k,v in students["housing_clean"].value_counts().items()})
except Exception:
    print("❌ A2 not yet — housing_clean should have exactly 3 groups (expect 590 / 945 / 492)")

✅ A2 correct — 3 groups: {'off-campus': 945, 'on-campus': 590, 'with family': 492}


### A3 · Errors vs. extremes
Find the impossible values. Store the sorted unique impossible ages as **`impossible_ages`** and the number of rows with negative work hours as **`n_neg_work`**. (Remember: extreme-but-valid values like a long commute are *kept*.)
*Hint:* boolean masks on `age` and `work_hours_per_week`.

In [ ]:
# TODO
impossible_ages = sorted(
    students.loc[(students["age"] < 0) | (students["age"] > 120), "age"].unique()
)

n_neg_work = (students["work_hours_per_week"] < 0).sum()

In [ ]:
# ✅ Check
try:
    assert 220 in impossible_ages and -22 in impossible_ages and n_neg_work == 4
    print("✅ A3 correct — impossible ages incl. -22/199/220; 4 negative work-hour rows")
except Exception:
    print("❌ A3 not yet — check ages (e.g. -22, 199, 220) and count negative work hours (expect 4)")

✅ A3 correct — impossible ages incl. -22/199/220; 4 negative work-hour rows


### A4 · Duplicates
Remove duplicate **student** records and store the result as **`students_dedup`**. Then, in the markdown cell after your code, explain in one sentence why you must NOT de-duplicate `enroll` or `activity` by ID.
*Hint:* `.drop_duplicates()` — think about exact vs. near-duplicates.

In [ ]:
# TODO: build students_dedup (one row per student)
students_dedup = students.drop_duplicates(
    subset="student_id",
    keep="first"
).copy()

**Why not de-dupe enroll / activity? (1 sentence):** I did not de-duplicate enroll or activity by student ID because each student can have multiple enrollment and weekly activity records.

In [ ]:
# ✅ Check
try:
    assert len(students_dedup) == 2000 and students_dedup["student_id"].is_unique
    print("✅ A4 correct — 2000 unique students (from 2027 rows)")
except Exception:
    print("❌ A4 not yet — students_dedup should be 2000 rows, one per student")

✅ A4 correct — 2000 unique students (from 2027 rows)


## Part B · Integrate the three files

### B5 · Standardize the key & integrate
Build one **row-per-student** analysis table called **`analysis`**: start from `students_dedup`, add a standardized numeric key, and merge in a per-student summary of `activity` (e.g., total `minutes_active`).
*Hint:* make the key with `.str.replace("NU-","")` → `int`; summarize activity with `groupby(...).sum()`; then `merge`.

In [ ]:
# TODO: build the standardized key and the one-row-per-student "analysis" table
# Create standardized numeric keys
students_dedup["student_key"] = (
    students_dedup["student_id"]
    .str.replace("NU-", "", regex=False)
    .astype(int)
)

activity["student_key"] = (
    activity["student_id"]
    .str.replace("NU-", "", regex=False)
    .astype(int)
)

# Summarize activity for each student
activity_summary = (
    activity.groupby("student_key", as_index=False)["minutes_active"]
    .sum()
    .rename(columns={"minutes_active": "total_minutes_active"})
)

# Merge with the student table
analysis = students_dedup.merge(
    activity_summary,
    on="student_key",
    how="left"
)

In [ ]:
# ✅ Check
try:
    assert len(analysis) == 2000 and analysis["student_id"].is_unique
    print("✅ B5 correct — one row per student, 2000 rows")
except Exception:
    print("❌ B5 not yet — analysis should have one row per student (2000)")

✅ B5 correct — one row per student, 2000 rows


### B6 · Verify the join
Report how many `enroll` rows match a student in your standardized key. Store the count as **`matched`**.
*Hint:* `enroll["sid"].isin(set_of_keys).sum()`

In [ ]:
# TODO
matched = enroll["sid"].isin(set(students_dedup["student_key"])).sum()

In [ ]:
# ✅ Check
try:
    assert matched == 8041
    print("✅ B6 correct — 8041 of 8088 enrollment rows match (14 orphan IDs)")
except Exception:
    print("❌ B6 not yet — count enrollment rows whose sid is in your student keys (expect 8041)")

✅ B6 correct — 8041 of 8088 enrollment rows match (14 orphan IDs)


## Part C · Transform

### C7 · Parse the dates
Parse `enrollment_date` so no valid date is lost. Store the parsed series as **`dates_parsed`** and check the number of NaT (blanks).
*Hint:* `pd.to_datetime(..., format="mixed", errors="coerce")` — compare NaT before and after.

In [ ]:
# TODO
dates_parsed = pd.to_datetime(
    students_dedup["enrollment_date"],
    format="mixed",
    errors="coerce"
)

In [ ]:
# ✅ Check
try:
    assert dates_parsed.isna().sum() == 0
    print("✅ C7 correct — all dates parsed, 0 lost (a naive parse would lose ~1470)")
except Exception:
    print("❌ C7 not yet — parse every format so no valid date becomes NaT")

✅ C7 correct — all dates parsed, 0 lost (a naive parse would lose ~1470)


### C8 · Normalize & discretize
Add two columns to `analysis`: a **z-scored** numeric column stored as **`analysis["study_z"]`**, and a **GPA band** column stored as **`analysis["gpa_band"]`** (bin `final_gpa` into 4 bands).
*Hint:* z-score = `(x - x.mean()) / x.std()`; bands = `pd.cut(..., bins=[-0.01,1,2,3,4])`.

In [ ]:
# TODO: add analysis["study_z"] and analysis["gpa_band"]
# Z-score study hours
analysis["study_z"] = (
    analysis["study_hours_reported"]
    - analysis["study_hours_reported"].mean()
) / analysis["study_hours_reported"].std()

# Divide GPA into 4 bands
analysis["gpa_band"] = pd.cut(
    analysis["final_gpa"],
    bins=[-0.01, 1, 2, 3, 4]
)

In [ ]:
# ✅ Check
try:
    assert analysis["gpa_band"].nunique() == 4 and abs(analysis["study_z"].mean()) < 0.01
    print("✅ C8 correct — z-score (mean ≈ 0) and 4 GPA bands added")
except Exception:
    print("❌ C8 not yet — add a z-scored column and a 4-band gpa_band column")

✅ C8 correct — z-score (mean ≈ 0) and 4 GPA bands added


## Part D · Deliver & reflect

### D9 · Write the clean file
Write your clean `analysis` table to **`northgate_clean.csv`** (do NOT overwrite the raw files).
*Hint:* `.to_csv("northgate_clean.csv", index=False)`

In [ ]:
# TODO: write analysis to northgate_clean.csv
analysis.to_csv("northgate_clean.csv", index=False)

In [ ]:
# ✅ Check
try:
    assert os.path.exists("northgate_clean.csv")
    print("✅ D9 correct — northgate_clean.csv written (raw files untouched)")
except Exception:
    print("❌ D9 not yet — write analysis to northgate_clean.csv")

✅ D9 correct — northgate_clean.csv written (raw files untouched)


### D10 · Cleaning log
In the markdown cell below, list each decision you made above and a one-line justification for it (missing values, housing, impossible values, duplicates, key, dates). *This is graded — no code needed.*

**Your cleaning log:**
- _decision → justification_
Missing values: I kept the missing study hours and reported the available sample size because filling them could make the data misleading.

Housing: I standardized the different housing spellings into three groups so the same categories are not counted separately.

Impossible values: I identified impossible ages (-22, 199, and 220) and 4 negative work-hour values as errors, while keeping extreme but possible values.

Duplicates: I removed duplicate student IDs so the data has one row per student, reducing 2,027 rows to 2,000 students.

Key: I removed the NU- prefix and created a numeric student key so the files could be matched correctly.

Join: I found that 8,041 of 8,088 enrollment rows matched the student records, with 14 orphan IDs.

Dates: I parsed the mixed enrollment date formats so no valid dates were lost.

Transformation: I created a z-score for study hours and divided GPA into four bands for easier analysis.

### D11 · Payoff
Using your clean `analysis` table, report the **mean GPA** and **one relationship** you find interesting, then note in one sentence how cleaning changed the picture versus the raw data.

In [ ]:
# TODO: compute the mean GPA and explore one relationship on the CLEAN data
# Mean GPA
mean_gpa = analysis["final_gpa"].mean()
print("Mean GPA:", round(mean_gpa, 2))

# Relationship: GPA by housing group
housing_gpa = analysis.groupby("housing_clean")["final_gpa"].mean()
print("\nMean GPA by housing:")
print(housing_gpa)

# Compare raw and clean GPA
raw_mean_gpa = students["final_gpa"].mean()
print("\nRaw mean GPA:", round(raw_mean_gpa, 2))
print("Clean mean GPA:", round(mean_gpa, 2))

Mean GPA: 2.31

Mean GPA by housing:
housing_clean
off-campus     2.329925
on-campus      2.276164
with family    2.293285
Name: final_gpa, dtype: float64

Raw mean GPA: 2.31
Clean mean GPA: 2.31


In [ ]:
# ✅ Check
print("(D11 is interpreted by your instructor — make sure your numbers and one-sentence takeaway are shown above.)")

(D11 is interpreted by your instructor — make sure your numbers and one-sentence takeaway are shown above.)


Mean GPA: The mean GPA is 2.31.

Relationship: Off-campus students had the highest average GPA at 2.33, followed by students living with family at 2.29 and on-campus students at 2.28. The differences are small, so housing does not show a strong relationship with GPA.

Effect of cleaning: The mean GPA stayed about the same at 2.31, but cleaning removed duplicates and standardized the data, making the results more consistent and reliable.